---

# PTA_Replicator Simulations

---

## Python Setup

General Python imports

In [1]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

NANOGrav-specific imports

In [2]:
from pta_replicator.white_noise import add_measurement_noise
from pta_replicator.white_noise import add_jitter
from pta_replicator.red_noise import add_red_noise, add_gwb
from pta_replicator.simulate import load_from_directories, make_ideal
import pint
pint.logging.setup(sink=sys.stderr, level="WARNING", usecolors=True)

libstempo not installed. PINT or libstempo are required to use par and tim files.


1

## CHANGEME: Simulation Parameters

General simulation parameters

In [3]:
N_PSR = 67
AMP = -15.
GAMMA = 13./3.
GAMMA_STR = r"$\frac{13}{3}$"

Random seeds

In [4]:
SEED_EFAC_EQUAD = 10660
SEED_JITTER = 17763
SEED_RED = 19870
SEED_GWB = 16672

File locations

In [5]:
PAR_DIR = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/data/NG15/par/"
TIM_DIR = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/data/NG15/tim/"
NOISE_DICT = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/data/NG15/ng15_dict.json"
NOISE_DICT_SAVE = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/data/NG15/ng15_sim_dict.json"
PLOT_DIR = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/results/A_15__gamma_13_3/plots/"
OUTPAR_DIR = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/results/A_15__gamma_13_3/par/"
OUTTIM_DIR = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/results/A_15__gamma_13_3/tim/"
OUTPAR_NOGWB_DIR = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/results/no_gwb/par/"
OUTTIM_NOGWB_DIR = "/home/catlettv/novus/sandbox/repos/VIPER/VIPER-2026/results/no_gwb/tim/"

Colors for plotting

In [6]:
C_DATA = "black"
C_GWB = "purple"
C_RED = "red"
C_MEAS = "green"
C_JITTER = "blue"

## Simulate the data

### Load `par` and `tim` files

In [7]:
psrs = load_from_directories(PAR_DIR, TIM_DIR, num_psrs=N_PSR)

WARNING  (pint.logging                  ): /home/catlettv/.conda/envs/viper-env-3.11/lib/python3.11/site-packages/pint/models/stand_alone_psr_binaries/DDK_model.py:146 UserWarning: DDK model uses KIN as inclination angle. SINI will not be used. This happens every time a DDK model is constructed.


### Load the noise dictionary
(Median values for the NANOGrav 15 year dataset)

In [8]:
with open(NOISE_DICT, 'r') as fp:
    noise_params = json.load(fp)
# change number strings to floats:
for value in noise_params.values():
    value = float(value)

### Parse the noise dictionary

In [9]:
psrlist = [psr.name for psr in psrs]
noise_dict = {}
for p in tqdm(psrlist):
    noise_dict[p] = {}
    noise_dict[p]['log10_equads'] = []
    noise_dict[p]['efacs'] = []
    noise_dict[p]['log10_ecorrs'] = []
    for ky in list(noise_params.keys()):
        if p in ky:
            if 'equad' in ky:
                noise_dict[p]['log10_equads'].append([ky.replace(p + '_' , '').replace('_log10_t2equad', ''), noise_params[ky]])
            if 'efac' in ky:
                noise_dict[p]['efacs'].append([ky.replace(p + '_' , '').replace('_efac', ''), noise_params[ky]])
            if 'ecorr' in ky:
                noise_dict[p]['log10_ecorrs'].append([ky.replace(p + '_' , '').replace('_log10_ecorr', ''), noise_params[ky]])
            if 'gamma' in ky:
                noise_dict[p]['rn_gamma'] = noise_params[ky]
            if 'log10_A' in ky:
                noise_dict[p]['rn_log10_amp'] = noise_params[ky]
                
    noise_dict[p]['log10_equads'] = np.array(noise_dict[p]['log10_equads'])
    noise_dict[p]['efacs'] = np.array(noise_dict[p]['efacs'])
    noise_dict[p]['log10_ecorrs'] = np.array(noise_dict[p]['log10_ecorrs'])

  0%|                                                                                           | 0/67 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████████████████| 67/67 [00:00<00:00, 6245.96it/s]

Save dictionary for later use

In [10]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        # Converts arrays to lists so JSON can serialize
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.generic):
            return obj.item()
        return super(NumpyEncoder, self).default(obj)

In [11]:
with open(NOISE_DICT_SAVE, "w") as file:
    json.dump(noise_dict, file, indent=4, cls=NumpyEncoder)

### Add noise components to the pulsars

In [12]:
for ii, psr in tqdm(enumerate(psrs)):

    ## make ideal
    make_ideal(psr)

    ## add efacs
    ## if you use flags, the flags and efac and equad values all need to have the same number of elements
    add_measurement_noise(psr, efac = noise_dict[psr.name]['efacs'][:,1].astype(float),
                          log10_equad = noise_dict[psr.name]['log10_equads'][:,1].astype(float), 
                          flagid = 'f', flags = noise_dict[psr.name]['efacs'][:,0], 
                          seed = SEED_EFAC_EQUAD + ii)

    ## add jitter
    add_jitter(psr, log10_ecorr = noise_dict[psr.name]['log10_ecorrs'][:,1].astype(float), 
                flagid='f', flags = noise_dict[psr.name]['log10_ecorrs'][:,0], 
                coarsegrain = 1.0/86400.0, seed = SEED_JITTER + ii)

    ## add red noise
    add_red_noise(psr,
                  log10_amplitude = noise_dict[psr.name]['rn_log10_amp'],
                  spectral_index = noise_dict[psr.name]['rn_gamma'],
                  components = 30, seed = SEED_RED + ii)

    ## save files
    outpar = OUTPAR_NOGWB_DIR + f"{psr.name}.par"
    outtim = OUTTIM_NOGWB_DIR + f"{psr.name}.tim"
    psr.write_partim(outpar=outpar, outtim=outtim)

0it [00:00, ?it/s]

1it [00:56, 56.68s/it]

2it [03:47, 124.11s/it]

3it [04:25, 84.73s/it] 

4it [06:26, 98.75s/it]

...

64it [1:21:04, 52.13s/it]

65it [1:22:15, 57.89s/it]

66it [1:24:06, 73.72s/it]

67it [1:24:29, 58.51s/it]

67it [1:24:29, 75.66s/it]

### Inject the GWB with given A and $\gamma$

In [13]:
add_gwb(psrs, log10_amplitude = AMP, spectral_index = GAMMA, seed = SEED_GWB)

### Plot the simulated residuals for each pulsar

In [14]:
for psr in tqdm(psrs):
    # Init plot
    fig, ax = plt.subplots(layout="constrained", figsize=(12, 4))

    # Names of added signals
    sig_meas = f"{psr.name}_measurement_noise"
    sig_jitter = f"{psr.name}_jitter"
    sig_red = f"{psr.name}_red_noise"
    sig_gwb = f"{psr.name}_gwb"

    # Horizontal line at 0
    plt.axhline(y=0, ls="--", color="gray")

    # Plot the data
    plt.errorbar(psr.toas.get_mjds(), psr.residuals.time_resids.to_value("us"),
                 psr.residuals.get_data_error().to_value("us"), c=C_DATA, marker="+", ls="", label="Total")

    # Plot individual signals
    plt.plot(psr.toas.get_mjds(), psr.added_signals_time[sig_meas].to_value("us"),
                     marker='x', label="Measurement Noise", color=C_MEAS, ls="", zorder=5, alpha=0.5)
    plt.plot(psr.toas.get_mjds(), psr.added_signals_time[sig_jitter].to_value("us"),
                         marker='x', label="Jitter", color=C_JITTER, ls="", zorder=5, alpha=0.5)
    plt.plot(psr.toas.get_mjds(), psr.added_signals_time[sig_red].to_value("us"),
                         marker='x', label="Red Noise", color=C_RED, ls="", zorder=5, alpha=0.5)
    plt.plot(psr.toas.get_mjds(), psr.added_signals_time[sig_gwb].to_value("us"),
                         marker='x', label="GWB", color=C_GWB, ls="", zorder=5, alpha=0.5)

    # Add a legend
    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width * 0.9, box.height])
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel("MJD")
    plt.ylabel(r"Residuals ($\mu s$)")
    plt.title(f"{psr.name} Simulated Residuals (A={AMP:.1f}, " + r"$\gamma$=" + GAMMA_STR + ")")
    plt.savefig(PLOT_DIR+f"{psr.name}.png")
    plt.clf()
    plt.close()

  0%|                                                                                           | 0/67 [00:00<?, ?it/s]

  1%|█▏                                                                                 | 1/67 [00:01<01:13,  1.12s/it]

  3%|██▍                                                                                | 2/67 [00:03<01:51,  1.71s/it]

  4%|███▋                                                                               | 3/67 [00:03<01:20,  1.26s/it]

  6%|████▉                                                                              | 4/67 [00:05<01:23,  1.33s/it]

...

 96%|██████████████████████████████████████████████████████████████████████████████▎   | 64/67 [01:16<00:02,  1.28it/s]

 97%|███████████████████████████████████████████████████████████████████████████████▌  | 65/67 [01:17<00:01,  1.19it/s]

 99%|████████████████████████████████████████████████████████████████████████████████▊ | 66/67 [01:19<00:00,  1.04it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 67/67 [01:19<00:00,  1.19it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 67/67 [01:19<00:00,  1.19s/it]

### Save the new `par` and `tim` files

In [15]:
for psr in tqdm(psrs):
    outpar = OUTPAR_DIR + f"{psr.name}.par"
    outtim = OUTTIM_DIR + f"{psr.name}.tim"
    psr.write_partim(outpar=outpar, outtim=outtim)

  0%|                                                                                           | 0/67 [00:00<?, ?it/s]

  1%|█▏                                                                                 | 1/67 [00:08<09:18,  8.46s/it]

  3%|██▍                                                                                | 2/67 [00:33<19:27, 17.96s/it]

  4%|███▋                                                                               | 3/67 [00:38<13:07, 12.31s/it]

  6%|████▉                                                                              | 4/67 [00:55<14:58, 14.27s/it]

...

 96%|██████████████████████████████████████████████████████████████████████████████▎   | 64/67 [12:15<00:23,  7.71s/it]

 97%|███████████████████████████████████████████████████████████████████████████████▌  | 65/67 [12:27<00:17,  8.80s/it]

 99%|████████████████████████████████████████████████████████████████████████████████▊ | 66/67 [12:42<00:10, 10.69s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 67/67 [12:45<00:00,  8.56s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 67/67 [12:45<00:00, 11.43s/it]